In [7]:
import os
import warnings
from IPython.display import display, Markdown, HTML
from dotenv import load_dotenv

from openai import OpenAI
import google.generativeai as genai
from anthropic import Anthropic

#load environment variables
load_dotenv()
print("Attempting to load API keys from .env file...")

#load keys
openai_api_key = os.getenv("OPENAI_API_KEY")
gemini_api_key = os.getenv("GEMINI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

#Configure clients
openai_client = OpenAI(api_key= openai_api_key)
print(f"OpenAI Client configured (Key starts with: {openai_api_key[:5]}...)")

genai.configure(api_key= gemini_api_key)
gemini_model = genai.GenerativeModel("gemini-2.5-flash")
print(f"Google Gemini Client configured (Key starts with: {gemini_api_key[:5]}...). Model: gemini-2.5-flash")


claude_client = Anthropic(api_key= anthropic_api_key)
print(f"Anthropic Claude Client configured (Key starts with: {anthropic_api_key[:5]}...)")

Attempting to load API keys from .env file...
OpenAI Client configured (Key starts with: sk-pr...)
Google Gemini Client configured (Key starts with: AIzaS...). Model: gemini-2.5-flash
Anthropic Claude Client configured (Key starts with: sk-an...)


In [2]:
def print_markdown(text):
    display(Markdown(text))

In [3]:
def display_html_code(provider_name, html_content):
    print_markdown(f"### Generated HTML from {provider_name}:")
    display(Markdown(f"'''html\n{html_content}\n'''"))

In [4]:
#let's test the Math capabilities of these 3 LLMs
test_prompt= "A father is 36 years old, and his son is 6 years old. In how many years will the father be exactly five times as old as his son?"

In [13]:
#let's test their creativity!
test_prompt = "Viết 1 bài thơ thú vị cho bạn của tôi khi mới 1 tuổi."

In [ ]:
response = openai_client.chat.completions.create(model = "gpt-4o",
                                                messages = [{"role": "user",
                                                            "content": test_prompt}],
                                                temperature = 0.5)
print_markdown(response.choices[0].message.content)

In [14]:
response = gemini_model.generate_content(test_prompt)
print_markdown(response.text)

Tuyệt vời! Chúc mừng sinh nhật bé nhé. Đây là một bài thơ vui vẻ dành tặng em bé tròn 1 tuổi:

**Em Bé Tròn Một Tuổi**

Hôm nay em bé tròn một tuổi rồi,
Cả nhà mình vui lắm, rộn tiếng cười!
Chân xinh chập chững bước đi đầu,
Mắt tròn xoe nhìn ngắm bao điều.

Miệng chúm chím bi bô, ê a gọi,
Cái tay bé xíu muốn chạm thật vội.
Đồ chơi nào cũng thích, cũng mê,
Khám phá bao điều hay, khắp bốn bề.

Chúc bé ăn ngoan, ngủ thật say,
Mỗi ngày khôn lớn, thêm điều hay!
Mong bé khỏe mạnh, thông minh, lanh lợi,
Chúc mừng sinh nhật, cả nhà yêu ơi!

---

**Giải thích một chút:**

*   **Hình ảnh sinh động:** Chú trọng các hành động và đặc điểm của bé 1 tuổi như chập chững đi, bi bô nói, tò mò khám phá.
*   **Ngôn ngữ đơn giản, vui tươi:** Dễ hiểu và có vần điệu để khi đọc lên nghe rất dễ thương.
*   **Lời chúc ý nghĩa:** Bao gồm các lời chúc về sức khỏe, sự phát triển và trí tuệ.
*   **Kết thúc ấm áp:** Thể hiện tình yêu thương của gia đình và bạn bè.

In [12]:
from anthropic import BadRequestError

try:
    response = claude_client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1000,
        messages=[{"role": "user", "content": test_prompt}]
    )
    print_markdown(response.content[0].text)

except BadRequestError as e:
    print("Claude API error:", e)
    print("Check Anthropic billing/credits, or use Gemini for now.")

C:\Users\PC\AppData\Local\Temp\ipykernel_6136\2349566972.py:4: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = claude_client.messages.create(


Claude API error: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CbMAPFPut7yiRaydsDHs1'}
Check Anthropic billing/credits, or use Gemini for now.


In [16]:
# Define the startup name and concept
startup_name = "ConnectGenius"
startup_concept = "An intelligent CRM system that uses AI to analyze customer interactions, predict needs, and automate personalized follow-ups. Focus on improving customer retention and sales efficiency for businesses of all sizes."

In [17]:
# Define the core prompt for the LLMs
# We explicitly ask for HTML code and specify the file name 'index.html'
html_prompt = f"""
You are a helpful AI assistant acting as a front-end web developer.

Your task is to generate the complete HTML code for a simple, clean, and professional-looking landing page (index.html) for a new startup.

Startup Name: {startup_name}
Concept: {startup_concept}

Please generate ONLY the full HTML code, starting with <!DOCTYPE html> and ending with </html>.
Create a modern, visually appealing landing page with the following:

-- Don't include images in the code. Raw html code with inline css for styling.

1. A sleek header with the startup name in a bold, modern font and a compelling tagline
2. A hero section with a clear value proposition and call-to-action button
3. A features section highlighting 3-4 key benefits with icons or simple visuals
4. A "How it Works" section with numbered steps
5. A testimonials section with fictional customer quotes
6. A pricing section with at least two tiers
7. A professional footer with navigation links and social media icons

Use inline CSS for styling with a modern color palette (primary, secondary, and accent colors). 
Include responsive design elements, subtle animations, and whitespace for readability.
Emphasize AI capabilities, ease of use, and business benefits throughout the copy.
Focus on conversion-optimized marketing messages that highlight pain points and solutions.

Do not include any explanations before or after the code block. Just provide the raw HTML code.
"""

print_markdown("**Core Prompt defined for the LLMs:**")
print_markdown(f"> {html_prompt}")  # Print the start of the prompt to verify

**Core Prompt defined for the LLMs:**

> 
You are a helpful AI assistant acting as a front-end web developer.

Your task is to generate the complete HTML code for a simple, clean, and professional-looking landing page (index.html) for a new startup.

Startup Name: ConnectGenius
Concept: An intelligent CRM system that uses AI to analyze customer interactions, predict needs, and automate personalized follow-ups. Focus on improving customer retention and sales efficiency for businesses of all sizes.

Please generate ONLY the full HTML code, starting with <!DOCTYPE html> and ending with </html>.
Create a modern, visually appealing landing page with the following:

-- Don't include images in the code. Raw html code with inline css for styling.

1. A sleek header with the startup name in a bold, modern font and a compelling tagline
2. A hero section with a clear value proposition and call-to-action button
3. A features section highlighting 3-4 key benefits with icons or simple visuals
4. A "How it Works" section with numbered steps
5. A testimonials section with fictional customer quotes
6. A pricing section with at least two tiers
7. A professional footer with navigation links and social media icons

Use inline CSS for styling with a modern color palette (primary, secondary, and accent colors). 
Include responsive design elements, subtle animations, and whitespace for readability.
Emphasize AI capabilities, ease of use, and business benefits throughout the copy.
Focus on conversion-optimized marketing messages that highlight pain points and solutions.

Do not include any explanations before or after the code block. Just provide the raw HTML code.


In [ ]:
# Let's generate HTML using OpenAI API, we will use gpt-4o model

openai_html_output = "<!-- OpenAI generation not run or failed -->"

print_markdown("## Calling OpenAI API...")
try:
    response = openai_client.chat.completions.create(
        model = "gpt-4o",
        messages = [
            {"role": "user", "content": html_prompt}
        ],
        temperature = 0.5,
    )
    openai_html_output = response.choices[0].message.content

    if openai_html_output.strip().startswith("```html"):
        lines = openai_html_output.strip().splitlines()
        openai_html_output = "\n".join(lines[1:-1]).strip()
    else:
        openai_html_output = openai_html_output.strip()

    # Display the generated HTML code
    display_html_code("OpenAI (gpt-4o)", openai_html_output)

    # Let's Save the output to a file
    file_path = "openai_landing_page.html"

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(openai_html_output)
    print_markdown(f"Successfully saved OpenAI output to `{file_path}`")

except Exception as e:
    print_markdown(f"Error calling OpenAI API: {e}")
    openai_html_output = f"<!-- Error calling OpenAI API: {e} -->"

In [18]:
gemini_html_output = "<!-- Gemini generation not run or failed -->"

print_markdown("## Calling Google Gemini API...")
try:
    response = gemini_model.generate_content(
        html_prompt,
    )

    raw_output = response.text
    if raw_output.strip().startswith("```html"):
        # Remove the first line (```html) and the last line (```)
        lines = raw_output.strip().splitlines()
        gemini_html_output = "\n".join(lines[1:-1]).strip()
    else:
        gemini_html_output = raw_output.strip()

    # Display the generated HTML code
    display_html_code("Google Gemini (gemini-2.5-flash)", gemini_html_output)

    # --- Save the output to a file ---
    file_path = "gemini_landing_page.html"

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(gemini_html_output)
    print_markdown(f"Successfully saved Gemini output to `{file_path}`")

except Exception as e:
    print_markdown(f"Error calling Google Gemini API: {e}")
    gemini_html_output = f"<!-- Error calling Google Gemini API: {e} -->"


## Calling Google Gemini API...

### Generated HTML from Google Gemini (gemini-2.5-flash):

'''html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>ConnectGenius - AI-Powered CRM for Smarter Customer Relationships</title>
    <meta name="description" content="ConnectGenius is an intelligent CRM system using AI to analyze customer interactions, predict needs, and automate personalized follow-ups. Improve customer retention and sales efficiency for businesses of all sizes.">
    <meta name="keywords" content="AI CRM, intelligent CRM, customer retention, sales efficiency, AI customer service, personalized follow-ups, ConnectGenius, B2B CRM, sales automation, marketing automation">
    <meta name="author" content="ConnectGenius">
</head>
<body style="margin: 0; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; line-height: 1.6; color: #333; background-color: #f8f9fa;">

    <!-- Header Section -->
    <header style="background-color: #4a148c; color: #fff; padding: 20px 0; text-align: center; box-shadow: 0 2px 10px rgba(0,0,0,0.1);">
        <div style="max-width: 1200px; margin: 0 auto; display: flex; justify-content: space-between; align-items: center; padding: 0 20px; flex-wrap: wrap;">
            <div style="flex: 1; text-align: left; min-width: 250px;">
                <h1 style="margin: 0; font-size: 2.2em; font-weight: 700; letter-spacing: 1px;">ConnectGenius</h1>
                <p style="margin: 5px 0 0; font-size: 1.1em; opacity: 0.9;">AI-Powered CRM for Smarter Customer Relationships</p>
            </div>
            <nav style="flex: 1; text-align: right; min-width: 250px;">
                <a href="#features" style="color: #fff; text-decoration: none; margin-left: 30px; font-size: 1.05em; transition: color 0.3s ease;">Features</a>
                <a href="#how-it-works" style="color: #fff; text-decoration: none; margin-left: 30px; font-size: 1.05em; transition: color 0.3s ease;">How It Works</a>
                <a href="#testimonials" style="color: #fff; text-decoration: none; margin-left: 30px; font-size: 1.05em; transition: color 0.3s ease;">Testimonials</a>
                <a href="#pricing" style="color: #fff; text-decoration: none; margin-left: 30px; font-size: 1.05em; transition: color 0.3s ease;">Pricing</a>
            </nav>
        </div>
    </header>

    <!-- Hero Section -->
    <section style="background: linear-gradient(135deg, #4a148c 0%, #6a1b9a 100%); color: #fff; padding: 120px 20px; text-align: center;">
        <div style="max-width: 900px; margin: 0 auto;">
            <h2 style="font-size: 3.5em; margin-bottom: 20px; font-weight: 800; letter-spacing: -1px;">Transform Customer Relationships with AI</h2>
            <p style="font-size: 1.5em; margin-bottom: 40px; line-height: 1.5; opacity: 0.9;">
                ConnectGenius is the intelligent CRM that leverages AI to deeply understand your customers,
                predict their needs, and automate truly personalized follow-ups. Boost retention and skyrocket sales for businesses of all sizes.
            </p>
            <a href="#get-started" style="background-color: #ffc107; color: #4a148c; padding: 18px 40px; font-size: 1.2em; font-weight: bold; text-decoration: none; border-radius: 50px; transition: background-color 0.3s ease, transform 0.3s ease; display: inline-block;">
                Get Started Free
            </a>
        </div>
    </section>

    <!-- Features Section -->
    <section id="features" style="padding: 80px 20px; background-color: #fff; text-align: center;">
        <h2 style="font-size: 2.8em; margin-bottom: 60px; color: #4a148c; font-weight: 700;">Intelligent Features, Real Business Impact</h2>
        <div style="max-width: 1200px; margin: 0 auto; display: flex; flex-wrap: wrap; justify-content: center; gap: 30px;">

            <!-- Feature 1 -->
            <div style="flex: 1 1 300px; padding: 40px; background-color: #f8f9fa; border-radius: 12px; box-shadow: 0 4px 20px rgba(0,0,0,0.08); transition: transform 0.3s ease; text-align: center;">
                <span style="font-size: 3em; color: #6a1b9a; display: block; margin-bottom: 20px;">🤖</span>
                <h3 style="font-size: 1.8em; color: #4a148c; margin-bottom: 15px;">AI-Powered Insights</h3>
                <p style="font-size: 1.1em; color: #555;">
                    Our AI analyzes every customer interaction, predicting needs, identifying trends, and uncovering upsell opportunities you might miss. Make data-driven decisions.
                </p>
            </div>

            <!-- Feature 2 -->
            <div style="flex: 1 1 300px; padding: 40px; background-color: #f8f9fa; border-radius: 12px; box-shadow: 0 4px 20px rgba(0,0,0,0.08); transition: transform 0.3s ease; text-align: center;">
                <span style="font-size: 3em; color: #6a1b9a; display: block; margin-bottom: 20px;">✉️</span>
                <h3 style="font-size: 1.8em; color: #4a148c; margin-bottom: 15px;">Automated Personalization</h3>
                <p style="font-size: 1.1em; color: #555;">
                    Deliver timely, personalized follow-ups and content automatically, ensuring every customer feels uniquely valued and understood, boosting engagement.
                </p>
            </div>

            <!-- Feature 3 -->
            <div style="flex: 1 1 300px; padding: 40px; background-color: #f8f9fa; border-radius: 12px; box-shadow: 0 4px 20px rgba(0,0,0,0.08); transition: transform 0.3s ease; text-align: center;">
                <span style="font-size: 3em; color: #6a1b9a; display: block; margin-bottom: 20px;">📈</span>
                <h3 style="font-size: 1.8em; color: #4a148c; margin-bottom: 15px;">Boost Sales & Retention</h3>
                <p style="font-size: 1.1em; color: #555;">
                    Streamline your sales process, reduce churn with proactive engagement, and empower your team to focus on high-value tasks, closing more deals faster.
                </p>
            </div>

            <!-- Feature 4 -->
            <div style="flex: 1 1 300px; padding: 40px; background-color: #f8f9fa; border-radius: 12px; box-shadow: 0 4px 20px rgba(0,0,0,0.08); transition: transform 0.3s ease; text-align: center;">
                <span style="font-size: 3em; color: #6a1b9a; display: block; margin-bottom: 20px;">🎯</span>
                <h3 style="font-size: 1.8em; color: #4a148c; margin-bottom: 15px;">Actionable Recommendations</h3>
                <p style="font-size: 1.1em; color: #555;">
                    Receive smart suggestions for next best actions, helping your team make informed decisions that drive customer success, increase loyalty, and maximize revenue.
                </p>
            </div>

        </div>
    </section>

    <!-- How it Works Section -->
    <section id="how-it-works" style="padding: 80px 20px; background-color: #f0f2f5; text-align: center;">
        <h2 style="font-size: 2.8em; margin-bottom: 60px; color: #4a148c; font-weight: 700;">Simple Steps to Smarter CRM</h2>
        <div style="max-width: 900px; margin: 0 auto; display: flex; flex-direction: column; gap: 40px;">

            <!-- Step 1 -->
            <div style="display: flex; align-items: center; justify-content: center; flex-wrap: wrap; text-align: left;">
                <div style="width: 60px; height: 60px; background-color: #ffc107; color: #4a148c; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-size: 1.8em; font-weight: bold; margin-right: 30px; flex-shrink: 0; margin-bottom: 15px;">1</div>
                <div style="flex: 1; min-width: 280px;">
                    <h3 style="font-size: 1.8em; color: #4a148c; margin-bottom: 10px;">Connect Your Data</h3>
                    <p style="font-size: 1.1em; color: #555;">
                        Seamlessly integrate ConnectGenius with your existing CRM, email, chat, and sales tools. Our AI starts learning and adapting instantly.
                    </p>
                </div>
            </div>

            <!-- Step 2 -->
            <div style="display: flex; align-items: center; justify-content: center; flex-wrap: wrap; text-align: left;">
                <div style="width: 60px; height: 60px; background-color: #ffc107; color: #4a148c; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-size: 1.8em; font-weight: bold; margin-right: 30px; flex-shrink: 0; margin-bottom: 15px;">2</div>
                <div style="flex: 1; min-width: 280px;">
                    <h3 style="font-size: 1.8em; color: #4a148c; margin-bottom: 10px;">AI Analyzes & Predicts</h3>
                    <p style="font-size: 1.1em; color: #555;">
                        Our powerful AI engine gets to work, analyzing customer behavior, sentiment, and historical data to predict future needs, preferences, and potential churn risks.
                    </p>
                </div>
            </div>

            <!-- Step 3 -->
            <div style="display: flex; align-items: center; justify-content: center; flex-wrap: wrap; text-align: left;">
                <div style="width: 60px; height: 60px; background-color: #ffc107; color: #4a148c; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-size: 1.8em; font-weight: bold; margin-right: 30px; flex-shrink: 0; margin-bottom: 15px;">3</div>
                <div style="flex: 1; min-width: 280px;">
                    <h3 style="font-size: 1.8em; color: #4a148c; margin-bottom: 10px;">Automate Personalized Engagement</h3>
                    <p style="font-size: 1.1em; color: #555;">
                        Set up intelligent automated workflows for personalized emails, follow-ups, product recommendations, and offers, delivered at the perfect moment.
                    </p>
                </div>
            </div>

            <!-- Step 4 -->
            <div style="display: flex; align-items: center; justify-content: center; flex-wrap: wrap; text-align: left;">
                <div style="width: 60px; height: 60px; background-color: #ffc107; color: #4a148c; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-size: 1.8em; font-weight: bold; margin-right: 30px; flex-shrink: 0; margin-bottom: 15px;">4</div>
                <div style="flex: 1; min-width: 280px;">
                    <h3 style="font-size: 1.8em; color: #4a148c; margin-bottom: 10px;">Measure & Optimize</h3>
                    <p style="font-size: 1.1em; color: #555;">
                        Gain deep insights into campaign performance, customer satisfaction, and ROI. Continuously optimize your strategy with AI-driven recommendations to maximize impact.
                    </p>
                </div>
            </div>

        </div>
    </section>

    <!-- Testimonials Section -->
    <section id="testimonials" style="padding: 80px 20px; background-color: #fff; text-align: center;">
        <h2 style="font-size: 2.8em; margin-bottom: 60px; color: #4a148c; font-weight: 700;">What Our Customers Are Saying</h2>
        <div style="max-width: 1200px; margin: 0 auto; display: flex; flex-wrap: wrap; justify-content: center; gap: 30px;">

            <!-- Testimonial 1 -->
            <div style="flex: 1 1 300px; background-color: #f8f9fa; padding: 35px; border-radius: 12px; box-shadow: 0 4px 15px rgba(0,0,0,0.05); text-align: center;">
                <p style="font-size: 1.1em; font-style: italic; color: #555; margin-bottom: 25px;">
                    "ConnectGenius has revolutionized how we engage with our clients. The AI predictions are eerily accurate, leading to a 25% increase in retention rate within months!"
                </p>
                <p style="font-weight: bold; color: #4a148c; margin-bottom: 5px;">Sarah J. - Marketing Director</p>
                <p style="font-size: 0.9em; color: #777;">Innovate Solutions Inc.</p>
            </div>

            <!-- Testimonial 2 -->
            <div style="flex: 1 1 300px; background-color: #f8f9fa; padding: 35px; border-radius: 12px; box-shadow: 0 4px 15px rgba(0,0,0,0.05); text-align: center;">
                <p style="font-size: 1.1em; font-style: italic; color: #555; margin-bottom: 25px;">
                    "Our sales team is more efficient than ever. ConnectGenius automates the tedious follow-ups and identifies hot leads, allowing them to focus on closing deals. Highly recommend!"
                </p>
                <p style="font-weight: bold; color: #4a148c; margin-bottom: 5px;">Mark D. - Sales Manager</p>
                <p style="font-size: 0.9em; color: #777;">Global Tech Ventures</p>
            </div>

            <!-- Testimonial 3 -->
            <div style="flex: 1 1 300px; background-color: #f8f9fa; padding: 35px; border-radius: 12px; box-shadow: 0 4px 15px rgba(0,0,0,0.05); text-align: center;">
                <p style="font-size: 1.1em; font-style: italic; color: #555; margin-bottom: 25px;">
                    "The personalization capabilities are unmatched. Our customers feel truly understood, leading to better engagement and significantly higher conversion rates across our campaigns."
                </p>
                <p style="font-weight: bold; color: #4a148c; margin-bottom: 5px;">Emily R. - CEO</p>
                <p style="font-size: 0.9em; color: #777;">NextGen Strategies</p>
            </div>

        </div>
    </section>

    <!-- Pricing Section -->
    <section id="pricing" style="padding: 80px 20px; background-color: #f0f2f5; text-align: center;">
        <h2 style="font-size: 2.8em; margin-bottom: 60px; color: #4a148c; font-weight: 700;">Affordable Plans for Every Business</h2>
        <div style="max-width: 1000px; margin: 0 auto; display: flex; flex-wrap: wrap; justify-content: center; gap: 30px;">

            <!-- Starter Plan -->
            <div style="flex: 1 1 300px; background-color: #fff; padding: 40px; border-radius: 12px; box-shadow: 0 4px 20px rgba(0,0,0,0.08); text-align: center; border-top: 5px solid #6a1b9a;">
                <h3 style="font-size: 2.2em; color: #4a148c; margin-bottom: 15px;">Starter</h3>
                <p style="font-size: 1.1em; color: #777; margin-bottom: 25px;">Perfect for small teams and startups looking to optimize customer interactions.</p>
                <div style="font-size: 3.5em; font-weight: bold; color: #4a148c; margin-bottom: 20px;">
                    $49<span style="font-size: 0.5em; font-weight: normal; color: #777;">/month</span>
                </div>
                <ul style="list-style: none; padding: 0; margin-bottom: 40px; text-align: left; font-size: 1.1em; color: #555;">
                    <li style="margin-bottom: 10px; display: flex; align-items: center;"><span style="color: #6a1b9a; font-size: 1.5em; margin-right: 10px;">✓</span>AI Interaction Analysis</li>
                    <li style="margin-bottom: 10px; display: flex; align-items: center;"><span style="color: #6a1b9a; font-size: 1.5em; margin-right: 10px;">✓</span>Basic Prediction Engine</li>
                    <li style="margin-bottom: 10px; display: flex; align-items: center;"><span style="color: #6a1b9a; font-size: 1.5em; margin-right: 10px;">✓</span>Automated Email Follow-ups</li>
                    <li style="margin-bottom: 10px; display: flex; align-items: center;"><span style="color: #6a1b9a; font-size: 1.5em; margin-right: 10px;">✓</span>Up to 5 Users</li>
                    <li style="display: flex; align-items: center;"><span style="color: #6a1b9a; font-size: 1.5em; margin-right: 10px;">✓</span>Standard Support</li>
                </ul>
                <a href="#signup-starter" style="background-color: #6a1b9a; color: #fff; padding: 15px 30px; font-size: 1.1em; font-weight: bold; text-decoration: none; border-radius: 50px; transition: background-color 0.3s ease, transform 0.3s ease; display: inline-block;">
                    Choose Starter
                </a>
            </div>

            <!-- Professional Plan (Highlighted) -->
            <div style="flex: 1 1 300px; background-color: #fff; padding: 50px 40px; border-radius: 12px; box-shadow: 0 8px 30px rgba(0,0,0,0.15); text-align: center; border-top: 5px solid #ffc107; transform: scale(1.05);">
                <span style="background-color: #ffc107; color: #4a148c; padding: 8px 20px; border-radius: 50px; font-weight: bold; font-size: 0.9em; position: relative; top: -20px; display: inline-block;">Most Popular</span>
                <h3 style="font-size: 2.5em; color: #4a148c; margin-bottom: 15px;">Professional</h3>
                <p style="font-size: 1.2em; color: #777; margin-bottom: 25px;">Grow your business with advanced AI capabilities and dedicated support.</p>
                <div style="font-size: 4em; font-weight: bold; color: #4a148c; margin-bottom: 20px;">
                    $129<span style="font-size: 0.5em; font-weight: normal; color: #777;">/month</span>
                </div>
                <ul style="list-style: none; padding: 0; margin-bottom: 40px; text-align: left; font-size: 1.1em; color: #555;">
                    <li style="margin-bottom: 10px; display: flex; align-items: center;"><span style="color: #ffc107; font-size: 1.5em; margin-right: 10px;">✓</span>Everything in Starter</li>
                    <li style="margin-bottom: 10px; display: flex; align-items: center;"><span style="color: #ffc107; font-size: 1.5em; margin-right: 10px;">✓</span>Advanced Predictive Analytics</li>
                    <li style="margin-bottom: 10px; display: flex; align-items: center;"><span style="color: #ffc107; font-size: 1.5em; margin-right: 10px;">✓</span>Multi-channel Automation (Email, SMS)</li>
                    <li style="margin-bottom: 10px; display: flex; align-items: center;"><span style="color: #ffc107; font-size: 1.5em; margin-right: 10px;">✓</span>Unlimited Users</li>
                    <li style="display: flex; align-items: center;"><span style="color: #ffc107; font-size: 1.5em; margin-right: 10px;">✓</span>Priority Support & Onboarding</li>
                </ul>
                <a href="#signup-professional" style="background-color: #ffc107; color: #4a148c; padding: 18px 40px; font-size: 1.2em; font-weight: bold; text-decoration: none; border-radius: 50px; transition: background-color 0.3s ease, transform 0.3s ease; display: inline-block;">
                    Choose Professional
                </a>
            </div>

            <!-- Enterprise Plan -->
            <div style="flex: 1 1 300px; background-color: #fff; padding: 40px; border-radius: 12px; box-shadow: 0 4px 20px rgba(0,0,0,0.08); text-align: center; border-top: 5px solid #6a1b9a;">
                <h3 style="font-size: 2.2em; color: #4a148c; margin-bottom: 15px;">Enterprise</h3>
                <p style="font-size: 1.1em; color: #777; margin-bottom: 25px;">Custom solutions for large organizations with complex needs.</p>
                <div style="font-size: 3.5em; font-weight: bold; color: #4a148c; margin-bottom: 20px;">
                    Custom
                </div>
                <ul style="list-style: none; padding: 0; margin-bottom: 40px; text-align: left; font-size: 1.1em; color: #555;">
                    <li style="margin-bottom: 10px; display: flex; align-items: center;"><span style="color: #6a1b9a; font-size: 1.5em; margin-right: 10px;">✓</span>Everything in Professional</li>
                    <li style="margin-bottom: 10px; display: flex; align-items: center;"><span style="color: #6a1b9a; font-size: 1.5em; margin-right: 10px;">✓</span>Dedicated AI Models</li>
                    <li style="margin-bottom: 10px; display: flex; align-items: center;"><span style="color: #6a1b9a; font-size: 1.5em; margin-right: 10px;">✓</span>Advanced Security & Compliance</li>
                    <li style="margin-bottom: 10px; display: flex; align-items: center;"><span style="color: #6a1b9a; font-size: 1.5em; margin-right: 10px;">✓</span>API Access & Custom Integrations</li>
                    <li style="display: flex; align-items: center;"><span style="color: #6a1b9a; font-size: 1.5em; margin-right: 10px;">✓</span>24/7 Enterprise Support</li>
                </ul>
                <a href="#contact" style="background-color: #6a1b9a; color: #fff; padding: 15px 30px; font-size: 1.1em; font-weight: bold; text-decoration: none; border-radius: 50px; transition: background-color 0.3s ease, transform 0.3s ease; display: inline-block;">
                    Contact Sales
                </a>
            </div>

        </div>
    </section>

    <!-- Footer Section -->
    <footer style="background-color: #212529; color: #f8f9fa; padding: 60px 20px; text-align: center;">
        <div style="max-width: 1200px; margin: 0 auto;">
            <div style="display: flex; flex-wrap: wrap; justify-content: space-between; align-items: flex-start; margin-bottom: 40px;">
                <!-- Company Info / Logo -->
                <div style="flex: 1 1 250px; margin-bottom: 20px; text-align: left;">
                    <h3 style="font-size: 1.8em; color: #fff; margin-bottom: 15px; letter-spacing: 0.5px;">ConnectGenius</h3>
                    <p style="font-size: 0.95em; color: #bbb;">AI-Powered CRM for Smarter Customer Relationships. Improve retention, boost sales, and delight your customers with intelligence.</p>
                </div>

                <!-- Navigation Links -->
                <div style="flex: 1 1 180px; margin-bottom: 20px; text-align: left;">
                    <h4 style="font-size: 1.2em; color: #fff; margin-bottom: 15px;">Quick Links</h4>
                    <ul style="list-style: none; padding: 0;">
                        <li style="margin-bottom: 10px;"><a href="#" style="color: #bbb; text-decoration: none; font-size: 0.95em; transition: color 0.3s ease;">Home</a></li>
                        <li style="margin-bottom: 10px;"><a href="#features" style="color: #bbb; text-decoration: none; font-size: 0.95em; transition: color 0.3s ease;">Features</a></li>
                        <li style="margin-bottom: 10px;"><a href="#how-it-works" style="color: #bbb; text-decoration: none; font-size: 0.95em; transition: color 0.3s ease;">How It Works</a></li>
                        <li style="margin-bottom: 10px;"><a href="#pricing" style="color: #bbb; text-decoration: none; font-size: 0.95em; transition: color 0.3s ease;">Pricing</a></li>
                        <li style="margin-bottom: 10px;"><a href="#contact" style="color: #bbb; text-decoration: none; font-size: 0.95em; transition: color 0.3s ease;">Contact</a></li>
                    </ul>
                </div>

                <!-- Legal/Resources -->
                <div style="flex: 1 1 180px; margin-bottom: 20px; text-align: left;">
                    <h4 style="font-size: 1.2em; color: #fff; margin-bottom: 15px;">Resources</h4>
                    <ul style="list-style: none; padding: 0;">
                        <li style="margin-bottom: 10px;"><a href="#" style="color: #bbb; text-decoration: none; font-size: 0.95em; transition: color 0.3s ease;">Blog</a></li>
                        <li style="margin-bottom: 10px;"><a href="#" style="color: #bbb; text-decoration: none; font-size: 0.95em; transition: color 0.3s ease;">Support</a></li>
                        <li style="margin-bottom: 10px;"><a href="#" style="color: #bbb; text-decoration: none; font-size: 0.95em; transition: color 0.3s ease;">Privacy Policy</a></li>
                        <li style="margin-bottom: 10px;"><a href="#" style="color: #bbb; text-decoration: none; font-size: 0.95em; transition: color 0.3s ease;">Terms of Service</a></li>
                    </ul>
                </div>

                <!-- Social Media -->
                <div style="flex: 1 1 180px; margin-bottom: 20px; text-align: left;">
                    <h4 style="font-size: 1.2em; color: #fff; margin-bottom: 15px;">Follow Us</h4>
                    <div style="display: flex; gap: 15px;">
                        <a href="#" style="color: #bbb; font-size: 1.8em; text-decoration: none; transition: color 0.3s ease;" title="Facebook">FB</a>
                        <a href="#" style="color: #bbb; font-size: 1.8em; text-decoration: none; transition: color 0.3s ease;" title="Twitter">TW</a>
                        <a href="#" style="color: #bbb; font-size: 1.8em; text-decoration: none; transition: color 0.3s ease;" title="LinkedIn">LI</a>
                        <a href="#" style="color: #bbb; font-size: 1.8em; text-decoration: none; transition: color 0.3s ease;" title="Instagram">IG</a>
                    </div>
                </div>
            </div>

            <div style="border-top: 1px solid #333; padding-top: 30px; margin-top: 30px;">
                <p style="font-size: 0.9em; color: #bbb; margin: 0;">&copy; 2023 ConnectGenius. All rights reserved.</p>
            </div>
        </div>
    </footer>

</body>
</html>
'''

Successfully saved Gemini output to `gemini_landing_page.html`

In [ ]:
# Generate HTML using Anthropic Claude

claude_html_output = "<!-- Claude generation not run or failed -->"

print_markdown("## Calling Anthropic Claude API...")

claude_model_name = "claude-3-7-sonnet-20250219"
print_markdown(f"(Using model: {claude_model_name})")

try:
    response = claude_client.messages.create(
        model = claude_model_name,
        max_tokens = 20000,
        messages = [{"role": "user", "content": html_prompt}],
    )
    raw_output = response.content[0].text
    if raw_output.strip().startswith("```html"):
        lines = raw_output.strip().splitlines()
        claude_html_output = "\n".join(lines[1:-1]).strip()
    else:
        claude_html_output = raw_output.strip()

    # Display the generated HTML code
    display_html_code(f"Anthropic Claude ({claude_model_name})", claude_html_output)

    # --- Save the output to a file ---
    file_path = "claude_landing_page.html"

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(claude_html_output)
    print_markdown(f"Successfully saved Claude output to `{file_path}`")

except Exception as e:
    print_markdown(f"Error calling Anthropic Claude API: {e}")
    claude_html_output = f"<!-- Error calling Anthropic Claude API: {e} -->"